# 
RLCT Estimation of Multitask Sparse Parity

In [1]:
%pip install devinterp seaborn torchvision pickleshare wandb plotly einops scikit-learn
!git clone https://github.com/ucla-vision/entropy-sgd.git
%cd entropy-sgd
from python.optim import EntropySGD
%cd ..

Defaulting to user installation because normal site-packages is not writeable
  Using cached matplotlib-3.9.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.3 MB)
  Using cached cloudpickle-3.0.0-py3-none-any.whl (20 kB)

[notice] A new release of pip is available: 23.1.2 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
fatal: destination path 'entropy-sgd' already exists and is not an empty directory.
/gpfs/home1/bshaffrey/entropy-sgd
/gpfs/home1/bshaffrey


In [2]:
import numpy as np
import torch as t
import torch
import torch.nn as nn
import torch.optim as optim
import time
import torch.nn.functional as F
import einops
import random
from dataclasses import dataclass
import os
import copy
import wandb
from tqdm.notebook import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from python.optim import EntropySGD
from torch.utils.data import DataLoader
from collections import defaultdict
from itertools import islice, product
import random
import time
from pathlib import Path

from devinterp.optim.sgld import SGLD
from devinterp.optim.sgnht import SGNHT

PRIMARY, SECONDARY, TERTIARY, QUATERNARY, QUINARY, SENARY = sns.color_palette("muted")[:6]
PRIMARY_LIGHT, SECONDARY_LIGHT, TERTIARY_LIGHT, QUATERNARY_LIGHT, QUINARY_LIGHT, SENARY_LIGHT = sns.color_palette(
    "pastel"
)[:6]

print(len(sns.color_palette("pastel")))

10


In [3]:
class FastTensorDataLoader:
    """
    A DataLoader-like object for a set of tensors that can be much faster than
    TensorDataset + DataLoader because dataloader grabs individual indices of
    the dataset and calls cat (slow).
    """
    def __init__(self, *tensors, batch_size=32, shuffle=False):
        """
        Initialize a FastTensorDataLoader.

        :param *tensors: tensors to store. Must have the same length @ dim 0.
        :param batch_size: batch size to load.
        :param shuffle: if True, shuffle the data *in-place* whenever an
            iterator is created out of this object.

        :returns: A FastTensorDataLoader.
        """
        assert all(t.shape[0] == tensors[0].shape[0] for t in tensors)
        self.tensors = tensors

        self.dataset_len = self.tensors[0].shape[0]
        self.batch_size = batch_size
        self.shuffle = shuffle

        # Calculate # batches
        n_batches, remainder = divmod(self.dataset_len, self.batch_size)
        if remainder > 0:
            n_batches += 1
        self.n_batches = n_batches

    def __iter__(self):
        if self.shuffle:
            self.indices = torch.randperm(self.dataset_len, device=self.tensors[0].device)
        else:
            self.indices = None
        self.i = 0
        return self

    def __next__(self):
        if self.i >= self.dataset_len:
            raise StopIteration
        if self.indices is not None:
            indices = self.indices[self.i:self.i+self.batch_size]
            batch = tuple(torch.index_select(t, 0, indices) for t in self.tensors)
        else:
            batch = tuple(t[self.i:self.i+self.batch_size] for t in self.tensors)
        self.i += self.batch_size
        return batch

    def __len__(self):
        return self.n_batches


def get_batch(n_tasks, n, Ss, codes, sizes, device='cpu', dtype=torch.float32):
    """Creates batch. 

    Parameters
    ----------
    n_tasks : int
        Number of tasks.
    n : int
        Bit string length for sparse parity problem.
    Ss : list of lists of ints
        Subsets of [1, ... n] to compute sparse parities on.
    codes : list of int
        The subtask indices which the batch will consist of
    sizes : list of int
        Number of samples for each subtask
    device : str
        Device to put batch on.
    dtype : torch.dtype
        Data type to use for input x. Output y is torch.int64.

    Returns
    -------
    x : torch.Tensor
        inputs
    y : torch.Tensor
        labels
    """
    batch_x = torch.zeros((sum(sizes), n_tasks+n), dtype=dtype, device=device)
    batch_y = torch.zeros((sum(sizes),), dtype=torch.int64, device=device)
    start_i = 0
    for (S, size, code) in zip(Ss, sizes, codes):
        if size > 0:
            x = torch.randint(low=0, high=2, size=(size, n), dtype=dtype, device=device)
            y = torch.sum(x[:, S], dim=1) % 2
            x_task_code = torch.zeros((size, n_tasks), dtype=dtype, device=device)
            x_task_code[:, code] = 1
            x = torch.cat([x_task_code, x], dim=1)
            batch_x[start_i:start_i+size, :] = x
            batch_y[start_i:start_i+size] = y
            start_i += size
    return batch_x, batch_y
    
def cycle(iterable):
    while True:
        for x in iterable:
            yield x


In [4]:
class MLP(nn.Module):
    
    def __init__(self, activation, depth, width):
        super(MLP, self).__init__()
        
        if activation == 'ReLU':
            activation_fn = nn.ReLU
        elif activation == 'Tanh':
            activation_fn = nn.Tanh
        elif activation == 'Sigmoid':
            activation_fn = nn.Sigmoid
        else:
            assert False, f"Unrecognized activation function identifier: {activation}"

        # create model
        layers = []
        for i in range(depth):
            if i == 0:
                layers.append(nn.Linear(n_tasks + n, width))
                layers.append(activation_fn())
            elif i == depth - 1:
                layers.append(nn.Linear(width, 2))
            else:
                layers.append(nn.Linear(width, width))
                layers.append(activation_fn())
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x)

In [5]:
def get_subsets(n_tasks, n, k):
    Ss = []
    for _ in range(n_tasks * 10):
        S = tuple(sorted(list(random.sample(range(n), k))))
        if S not in Ss:
            Ss.append(S)
        if len(Ss) == n_tasks:
            break
    assert len(Ss) == n_tasks, "Couldn't find enough subsets for tasks for the given n, k"
    return Ss

def get_data(steps, batch_size, cdf, n_tasks, n, Ss, device, dtype, test_points_per_task):
    x, y = torch.zeros((steps * batch_size, n_tasks + n), dtype=dtype), torch.zeros((steps * batch_size), dtype=torch.int64)
    x_sub, y_sub = list(), list()
    for task in range(n_tasks):
        x_sub.append(torch.zeros((steps * test_points_per_task, n_tasks + n), dtype=dtype))
        y_sub.append(torch.zeros((steps * test_points_per_task), dtype=torch.int64))
    for step in tqdm(range(steps)):
        samples = np.searchsorted(cdf, np.random.rand(batch_size,))
        hist, _ = np.histogram(samples, bins=n_tasks, range=(0, n_tasks-1))
        x[batch_size * step : batch_size * (step + 1), : ], y[batch_size * step : batch_size * (step + 1)] = get_batch(n_tasks=n_tasks, n=n, Ss=Ss, codes=list(range(n_tasks)), sizes=hist, device=device, dtype=dtype)
        for task in range(n_tasks):
           x_sub[task][test_points_per_task * step : test_points_per_task * (step + 1), : ], y_sub[task][test_points_per_task * step : test_points_per_task * (step + 1)] = get_batch(n_tasks=n_tasks, n=n, Ss=[Ss[task]], codes=[task], sizes=[test_points_per_task], device=device, dtype=dtype)
    train_data = list(zip(x, y))
    train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
    train_loaders_subtasks = []

    for task in range(n_tasks):
        train_data_sub = list(zip(x_sub[task], y_sub[task]))
        train_loaders_subtasks.append(torch.utils.data.DataLoader(train_data_sub, batch_size=batch_size, shuffle=True))
    return train_loader, train_loaders_subtasks

In [25]:
def accuracy_function(outputs, targets):
    return (outputs.argmax(1) == targets).float().mean()
    #torch.sum(labels_i_pred == y_i).item() / test_points

def train_one_epoch(model, train_loader, train_loaders_subtasks, optimizer, criterion, device, n_tasks):
    model.train()
    losses_subtasks = []
    accuracies_subtasks = []
    coeff = 1.0
    for task in range(n_tasks):
        subtask_losses = 0
        subtask_accuracies = 0
        train_loss = 0
        train_accuracy = 0
        for (data_subtask, targets_subtask), (data, targets) in zip(train_loaders_subtasks[task], train_loader):
            optimizer.zero_grad()
            outputs = model(data.to(device))
            loss = criterion(outputs, targets.to(device))
            if task == n_tasks - 1:
                train_loss += loss.item()
                train_accuracy += accuracy_function(outputs, targets.to(device)).item()
            outputs_subtask = model(data_subtask.to(device))
            loss_subtask = criterion(outputs_subtask, targets_subtask.to(device))
            subtask_losses += loss_subtask.item()
            accuracy_subtask = accuracy_function(outputs_subtask, targets_subtask.to(device))
            subtask_accuracies += accuracy_subtask.item()
            #loss += coeff * loss_subtask
            loss.backward()
            optimizer.step()
        losses_subtasks.append(subtask_losses / len(train_loaders_subtasks[task]))
        accuracies_subtasks.append(subtask_accuracies / len(train_loaders_subtasks[task]))
        
    return train_loss / len(train_loader), train_accuracy / len(train_loader), losses_subtasks, accuracies_subtasks


def evaluate(model, test_loader, criterion):
    model.eval()
    test_loss = 0
    test_accuracy = 0
    with torch.no_grad():
        for index, (data, targets) in enumerate(test_loader):
            outputs = model(data.to(device))
            loss = criterion(outputs, targets.to(device))
            test_loss += loss.item()
            test_accuracy += accuracy_function(outputs, targets.to(device)).item()
            
    return test_loss / len(test_loader), test_accuracy / len(test_loader)


In [30]:
def l0_regulariser(parameters, alpha=1.0):
    loss = 0
    for param in parameters:
        loss += torch.sum(param != 0).float()
    return alpha * loss

def l1_loss_zero_one(parameters):
    l1_loss = 0
    for param in parameters:
        #print(param)
        #print(param.abs())
        #print((param - 1).abs())
        #print(torch.min(param.abs(), (param - 1).abs()))
        #print(torch.min(param.abs(), (param - 1).abs()).sum())
        #print('\n\n\n\n')
        l1_loss += torch.min(param.abs(), (param - 1).abs()).sum()
    return l1_loss

def double_well_regulariser(parameters, beta=1.0):
    loss = 0
    for param in parameters:
        loss += beta * (((param - .5)** 2 - .25) ** 2).sum()
    return loss

def run(n_tasks,
        n,
        k,
        D,
        width,
        depth,
        activation,
        test_points,
        test_points_per_task,
        steps,
        epochs,
        batch_size,
        lr,
        weight_decay,
        device,
        dtype,
        log_freq,
        verbose,
        seed):

    torch.set_default_dtype(dtype)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    info = {}
    models_saved = []

    mlp = MLP(activation, depth, width).to(device)
    info['P'] = sum(t.numel() for t in mlp.parameters())

    Ss = get_subsets(n_tasks, n, k)
    info['Ss'] = Ss

    probs = np.array([np.power(n, -alpha) for n in range(1+offset, n_tasks+offset+1)])
    probs = probs / np.sum(probs)
    cdf = np.cumsum(probs)

    test_batch_sizes = [int(prob * test_points) for prob in probs]

    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(mlp.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.98))
    
    info['accuracies'] = list()
    info['losses'] = list()
    info['losses_subtasks'] = dict()
    info['accuracies_subtasks'] = dict()
    for i in range(n_tasks):
        info['losses_subtasks'][str(i)] = list()
        info['accuracies_subtasks'][str(i)] = list()

    train_loader, train_loaders_subtasks = get_data(steps, batch_size, cdf, n_tasks, n, Ss, device, dtype, test_points_per_task)
        
    for epoch in tqdm(range(epochs), disable=not verbose):
        train_loss, train_accuracy, losses_subtasks, accuracies_subtasks = train_one_epoch(mlp, train_loader, train_loaders_subtasks, optimizer, loss_fn, device, n_tasks)
        if epoch % log_freq == 0:
            info['accuracies'].append(train_accuracy) 
            info['losses'].append(train_loss)
            for task in range(n_tasks):
                info['losses_subtasks'][str(task)].append(losses_subtasks[task])
                info['accuracies_subtasks'][str(task)].append(accuracies_subtasks[task])
            models_saved += [copy.deepcopy(mlp)]
        
    return info, models_saved, loss_fn, train_loader, train_loaders_subtasks, Ss

n_tasks = 5
n = 50
k = 3
alpha = 1.5
offset = 0

D = -1 # -1 for infinite data

width = 1000
depth = 2
activation = 'ReLU'
    
steps = 10
batch_size = 10000
lr = 1e-3
weight_decay = 1.0
test_points = 30000
test_points_per_task = 1000
epochs = 1000
stop_early = False
    
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
dtype = torch.float32

log_freq = 10
verbose=True
seed = 0
runs = 1
        
info, models_saved, criterion, train_loader, train_loaders_subtasks, Ss = run(n_tasks,
    n, 
    k, 
    D, 
    width, 
    depth, 
    activation, 
    test_points, 
    test_points_per_task, 
    steps,
    epochs,
    batch_size, 
    lr, 
    weight_decay, 
    device, 
    dtype, 
    log_freq, 
    verbose, 
    seed)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

In [31]:
Ss = get_subsets(n_tasks, n, k)
probs = np.array([np.power(n, -alpha) for n in range(1+offset, n_tasks+offset+1)])
probs = probs / np.sum(probs)
cdf = np.cumsum(probs)
test_batch_sizes = [int(prob * test_points) for prob in probs]

In [13]:
from devinterp.slt.sampler import estimate_learning_coeff_with_summary

N_EPOCHS = epochs
SAVE_EVERY_N_EPOCHS = log_freq

def estimate_rlcts(models, train_loader, criterion, data_length, device, num_draws, num_models):
    estimates = {"sgnht": [], "sgld": []}
    for idx, model in enumerate(tqdm(models)):
        for method, optimizer_kwargs in [
            #("sgnht", {"lr": 1e-7, "diffusion_factor": 0.01}),
            ("sgld", {"lr": 1e-3, "localization": 1000.0, "noise_level": 1.0}),
        ]:
            results = estimate_learning_coeff_with_summary(
                model,
                train_loader,
                criterion=criterion,
                optimizer_kwargs=optimizer_kwargs,
                sampling_method=SGNHT if method == "sgnht" else SGLD,
                num_chains=1,
                num_draws=num_draws,
                num_burnin_steps=100,
                num_steps_bw_draws=1,
                device=device,
                seed=0
            )
            estimate = results["llc/mean"]
            estimates[method].append(estimate)
    return estimates

def obtain_rlct_estimates(train_loader, models_saved, criterion, runs):
    num_models = N_EPOCHS // SAVE_EVERY_N_EPOCHS
    data_length = len(train_loader)
    rlct_estimates = {"sgnht": torch.zeros(runs, num_models), "sgld": torch.zeros(runs, num_models)}
    num_draws = 400

    for run in tqdm(range(runs)):
        rlct_estimate = estimate_rlcts(
            models_saved[num_models * run : num_models * (run + 1)], train_loader, criterion, data_length, device, num_draws, num_models
        )
        #rlct_estimates["sgnht"][run] = torch.tensor(rlct_estimate["sgnht"])
        rlct_estimates["sgld"][run] = torch.tensor(rlct_estimate["sgld"])

    rlct_estimates_final = {"sgnht": rlct_estimates["sgnht"].mean(dim=0), "sgld": rlct_estimates["sgld"].mean(dim=0)}
    return rlct_estimates_final

#rlct_estimates_final = obtain_rlct_estimates(train_loader, models_saved, criterion, runs)

In [10]:
from devinterp.slt.sampler import  sample, LLCEstimator
from devinterp.optim import SGLD
from devinterp.utils import default_nbeta

# Assuming you have a PyTorch Model assigned to model, and DataLoader assigned to trainloader
llc_estimator = LLCEstimator(..., nbeta=default_nbeta(train_loader))
sample(models_saved[-1], train_loader, ..., callbacks = [llc_estimator])

llc_mean = llc_estimator.get_results()["llc/mean"]
print(llc_mean)

TypeError: LLCEstimator.__init__() missing 2 required positional arguments: 'num_draws' and 'init_loss'

In [32]:
dataset = '0'

def plot_losses(train_losses_final, name = ''):

    sns.set_style("whitegrid")
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color=PRIMARY)
    plt.yscale('log')
    ax1.plot(x_axis, train_losses_final, label="Train Loss, sgd", color=PRIMARY)
    #ax1.plot(x_axis, test_losses_final, label="Test Loss, sgd", color=PRIMARY_LIGHT)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("losses_" + name + "_" + str(N_EPOCHS) + "_epochs.png")
    
def plot_subtask_losses(train_losses_subtasks, n_tasks, name='full'):

    sns.set_style("whitegrid")
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss", color=PRIMARY)
    plt.yscale('log')
    
    for task in range(n_tasks):
        ax1.plot(x_axis, train_losses_subtasks[str(task)], label="Task " + str(task), color=sns.color_palette("muted")[task])
        
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("losses_subtasks_" + name + "_" + str(N_EPOCHS) + "_epochs.png")
    
def plot_accuracies(train_accuracies_final, name = ''):

    sns.set_style("whitegrid")
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy", color=PRIMARY)
    plt.yscale('log')
    ax1.plot(x_axis, train_accuracies_final, label="Train Accuracy, sgd", color=PRIMARY)
    #ax1.plot(x_axis, test_accuracies_final, label="Test Accuracy, sgd", color=PRIMARY_LIGHT)
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("accuracies_" + name + "_" + str(N_EPOCHS) + "_epochs.png")
    
def plot_subtask_accuracies(train_accuracies_subtasks, n_tasks, name='full'):

    sns.set_style("whitegrid")
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax1 = plt.subplots(figsize=(10, 6))
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy", color=PRIMARY)
    plt.yscale('log')
    
    for task in range(n_tasks):
        ax1.plot(x_axis, train_accuracies_subtasks[str(task)], label="Task " + str(task), color=sns.color_palette("muted")[task])
        
    ax1.tick_params(axis="y", labelcolor=PRIMARY)
    ax1.legend(loc="upper left")
    fig.tight_layout()
    plt.show()
    fig.savefig("accuracies_subtasks_" + name + "_" + str(N_EPOCHS) + "_epochs.png")
    
def plot_rlcts(rlct_estimates_final, dataset, rlct_estimates_final_other = {}):

    sns.set_style("whitegrid")
    
    #first_part = np.arange(1, 1001, 10)
    
    # Create array from 1000 to 50000 with step 100
    # Start from 1100 to avoid duplicating 1000
    #second_part = np.arange(1001, N_EPOCHS, 100)
    
    # Combine the two arrays
    #x_axis = np.concatenate([first_part, second_part])
    x_axis = np.arange(1, N_EPOCHS, SAVE_EVERY_N_EPOCHS)

    fig, ax2 = plt.subplots(figsize=(10, 6))
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel(r"Local Learning Coefficient, $\hat \lambda$", color=SECONDARY)
    if rlct_estimates_final_other:
        ax2.plot(x_axis, rlct_estimates_final_other["sgld"], label="summed curve", color=TERTIARY)
    ax2.plot(x_axis, rlct_estimates_final["sgld"], label="SGLD, sgd", color=TERTIARY_LIGHT)
    ax2.tick_params(axis="y", labelcolor=SECONDARY)
    ax2.legend(loc="center right")

    fig.tight_layout()
    plt.show()
    fig.savefig("rclt_" + dataset + "_" + str(N_EPOCHS) + "_epochs.png")

train_losses_final = info['losses']
train_accuracies_final = info['accuracies']
    
plot_losses(train_losses_final, dataset)
plot_subtask_losses(info['losses_subtasks'], n_tasks)
plot_accuracies(train_accuracies_final, dataset)
plot_subtask_accuracies(info['accuracies_subtasks'], n_tasks)
#plot_rlcts(rlct_estimates_final, dataset='full')


/scratch-local/bshaffrey.7971899/ipykernel_2673775/3095880001.py:8: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax1 = plt.subplots(figsize=(10, 6))
/scratch-local/bshaffrey.7971899/ipykernel_2673775/3095880001.py:17: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/scratch-local/bshaffrey.7971899/ipykernel_2673775/3095880001.py:36: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/scratch-local/bshaffrey.7971899/ipykernel_2673775/3095880001.py:53: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/scratch-local/bshaff

In [32]:
def return_topk_percent_mask(tensor, proportion):
    # Step 1: Flatten the tensor
    flattened_tensor = tensor.flatten()

    # Step 2: Determine K, where K is 20% of the total number of elements
    total_elements = flattened_tensor.numel()
    K = int(proportion * total_elements)

    # Step 3: Find the value of the K-th largest element
    topk_values, _ = torch.topk(flattened_tensor, K)
    threshold_value = topk_values[-1]

    # Step 4: Create a boolean mask of the top K values
    return tensor >= threshold_value


def unravel_index(index, shape):
    out = []
    for dim in reversed(shape):
        out.append(index % dim)
        index = index // dim
    return tuple(reversed(out))

def ablation_study(model, loss_fn):
    
    loss_diffs_per_task = []
    for index in tqdm(range(n_tasks)):
        loss_diffs = {}
        for name, param in model.named_parameters():
            loss_diffs[name] = torch.zeros(param.shape)
            x_i, y_i = get_batch(n_tasks=n_tasks, n=n, Ss=[Ss[index]], codes=[index], sizes=[test_points_per_task], device=device, dtype=dtype)
            y_i_pred = model(x_i)
            loss_baseline = loss_fn(y_i_pred, y_i).item()
            
            for idx in tqdm(range(param.numel())):
                with torch.no_grad():
                    # Convert flat index i to multi-dimensional index for the original shape
                    multi_idx = unravel_index(idx, param.shape)
                
                    # Save the original weight value
                    original_value = param[multi_idx].item()
                
                    # Set the weight to zero
                    param[multi_idx] = 0.0
                    
                    y_i_ablated = model(x_i)
                    loss_ablated = loss_fn(y_i_ablated, y_i).item()
                    
                    loss_diffs[name][multi_idx] = abs(loss_ablated - loss_baseline)
                    
                    # Restore the original weight
                    param[multi_idx] = original_value
                    torch.cuda.empty_cache()
        loss_diffs_per_task.append(loss_diffs)
                    
    return loss_diffs_per_task   

# Use the function
loss_diffs_per_task = ablation_study(models_saved[-1], criterion)

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/165000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/6000 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/165000 [00:00<?, ?it/s]

  0%|          | 0/3000 [00:00<?, ?it/s]

  0%|          | 0/6000 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

In [33]:
indices_per_task = []
# Analyze results
for task in tqdm(range(n_tasks)):
    
    indices_dict = {}
    
    proportion = 0.1
    b_0 = loss_diffs_per_task[task]['model.0.bias'].shape[0]
    b_1 = loss_diffs_per_task[task]['model.2.bias'].shape[0]
    indices_dict['model.0.weight'] = return_topk_percent_mask(loss_diffs_per_task[task]['model.0.weight'], proportion)
    indices_dict['model.0.bias'] = torch.ones(b_0, dtype=torch.bool)
    indices_dict['model.2.weight'] = return_topk_percent_mask(loss_diffs_per_task[task]['model.2.weight'], proportion)
    indices_dict['model.2.bias'] = torch.ones(b_1, dtype=torch.bool)
        
    indices_per_task.append(indices_dict)

  0%|          | 0/5 [00:00<?, ?it/s]

In [34]:
def get_path_losses_per_task(loss_diffs_per_task, n_tasks):
    path_losses_per_task = []
    path_indices_per_task = []
    indices_per_task = []

    for task in tqdm(range(n_tasks)):
        W_0 = loss_diffs_per_task[task]['model.0.weight']
        b_0 = loss_diffs_per_task[task]['model.0.bias']
        W_1 = loss_diffs_per_task[task]['model.2.weight']
        b_1 = loss_diffs_per_task[task]['model.2.bias']
        
        indices_dict = {}
        
        indices_per_task.append(indices_dict)
    
        path_losses = W_0[ : , : , None, None] + W_1[None, None, : , : ]
        epsilon = 0.04
        indices =  path_losses > epsilon
        indices_dict['model.0.weight'] = indices.any(dim=(2, 3))
        indices_dict['model.0.bias'] = torch.ones(b_0.shape, dtype=torch.bool)
        indices_dict['model.2.weight'] = indices.any(dim=(0, 1))
        indices_dict['model.2.bias'] = torch.ones(b_1.shape, dtype=torch.bool)

        print(indices_dict['model.0.weight'])
        print(indices_dict['model.0.bias'])
        print(indices_dict['model.2.weight'])
        print(indices_dict['model.2.bias'])
        indices_per_task.append(indices_dict)
        path_losses_per_task.append(path_losses.flatten())
        
    return path_losses_per_task, indices_per_task

for name, param in models_saved[-1].named_parameters():
    print(name)
    print(param.shape)

#path_losses_per_task, indices_per_task = get_path_losses_per_task(loss_diffs_per_task, n_tasks)

model.0.weight
torch.Size([3000, 55])
model.0.bias
torch.Size([3000])
model.2.weight
torch.Size([2, 3000])
model.2.bias
torch.Size([2])


In [35]:
import copy

def prune_to_obtain_circuit(model, model_indices):
    
    model_state_dict = model.state_dict()
    
    for name, param in model.named_parameters():
        indices = model_indices[name]
        model_state_dict[name][~indices] = 0.0
        
    model.load_state_dict(model_state_dict)
            
    return model

models_per_task = []

for task in tqdm(range(n_tasks)):
    models = []
    for model in tqdm(models_saved):
        task_model = copy.deepcopy(model)
        task_model = prune_to_obtain_circuit(task_model, indices_per_task[task])
        models.append(task_model)
    models_per_task.append(models)

        

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

In [36]:
def compute_loss_curve_for_model(models, train_loader, train_loaders_subtasks, info, steps, n, Ss, n_tasks, test_points, test_batch_sizes, device, dtype):
    losses = []
    accuracies = []
    losses_subtasks = {}
    accuracies_subtasks = {}
    loss_fn = nn.CrossEntropyLoss()
    
    for i in range(n_tasks):
        losses_subtasks[str(i)] = list()
        accuracies_subtasks[str(i)] = list()

    for index, model in tqdm(enumerate(models)):
        train_loss = 0
        train_accuracy = 0
        coeff = 1.0
        for index, (data, targets) in enumerate(train_loader):
            outputs = model(data.to(device))
            loss = criterion(outputs, targets.to(device))
            train_loss += loss.item()
            train_accuracy += accuracy_function(outputs, targets.to(device)).item()
        for task in range(n_tasks):
            subtask_losses = 0
            subtask_accuracies = 0
            for (data_subtask, targets_subtask) in train_loaders_subtasks[task]:
                outputs_subtask = model(data_subtask.to(device))
                loss_subtask = criterion(outputs_subtask, targets_subtask.to(device))
                subtask_losses += loss_subtask.item()
                accuracy_subtask = accuracy_function(outputs_subtask, targets_subtask.to(device))
                subtask_accuracies += accuracy_subtask.item()
            losses_subtasks[str(task)].append(subtask_losses / len(train_loaders_subtasks[task]))
            accuracies_subtasks[str(task)].append(subtask_accuracies / len(train_loaders_subtasks[task]))
                    
        accuracies.append(train_accuracy / len(train_loader)) 
        losses.append(train_loss / len(train_loader))
    return losses, accuracies, losses_subtasks, accuracies_subtasks

for task in range(n_tasks):
    losses, accuracies, losses_subtasks, accuracies_subtasks = compute_loss_curve_for_model(models_per_task[task], train_loader, train_loaders_subtasks, info, steps, n, Ss, n_tasks, test_points, test_batch_sizes, device, dtype)
    
    print(len(losses), len(accuracies), len(losses_subtasks['0']))
    plot_losses(losses, 'task_'+ str(task) + '_with_ablation')
    plot_accuracies(accuracies, 'task_'+ str(task) + '_with_ablation')
    plot_subtask_losses(losses_subtasks, n_tasks, 'task_'+ str(task) + '_with_ablation')
    plot_subtask_accuracies(accuracies_subtasks, n_tasks, 'task_'+ str(task) + '_with_ablation')

0it [00:00, ?it/s]

200 200 200


/scratch-local/bshaffrey.7747159/ipykernel_2425459/3095880001.py:17: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/scratch-local/bshaffrey.7747159/ipykernel_2425459/3095880001.py:53: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/scratch-local/bshaffrey.7747159/ipykernel_2425459/3095880001.py:36: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()
/scratch-local/bshaffrey.7747159/ipykernel_2425459/3095880001.py:72: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


0it [00:00, ?it/s]

200 200 200


...

200 200 200


0it [00:00, ?it/s]

200 200 200


0it [00:00, ?it/s]

200 200 200
